# Shared Mimir and Baldr infrastructure

Mimir is now Skuld's time-series and power-spectrum engine. Skuld keeps a thin regular-grid adapter because its forward model needs the exact observing mask. Baldr supplies the Normal and Beta transforms used by the nuisance priors; Skuld keeps its array-parameter Gamma likelihood and multivariate Student proposal because those are not currently Baldr distribution APIs.

In [ ]:
import numpy as np
from mimir import TimeSeries
from asterodetect import PreparedTessLightCurve


In [ ]:
cadence_seconds = 120.0
sample_count = 1024
time = np.arange(sample_count) * cadence_seconds / 86400.0
flux = 20.0 * np.sin(2 * np.pi * 800e-6 * time * 86400.0)
observed = np.ones(sample_count, dtype=bool)
observed[300:330] = False
series = TimeSeries(time[observed], flux[observed], time_unit='d', flux_unit='ppm')
prepared = PreparedTessLightCurve.from_irregular(series.time, series.flux, cadence_seconds=cadence_seconds, flux_unit='ppm', sigma_clip=None)
spectrum = prepared.to_power_spectrum()


In [ ]:
peak_frequency = spectrum.frequency[np.argmax(spectrum.power)]
integrated_variance = np.sum(spectrum.power * spectrum.bins_averaged * (spectrum.bin_upper - spectrum.bin_lower))
print(f'Peak: {peak_frequency:.1f} microhertz')
print(f'Duty cycle: {prepared.observing_window.duty_cycle:.3f}')
print(f'Integrated PSD: {integrated_variance:.2f} ppm^2')
